In [1]:
# Метод 3: Fast‑DetectGPT на Qwen2.5‑0.5B
### Основан на логарифмической вероятности текста (чем выше, тем вероятнее AI)

!pip install -q torch transformers tqdm scikit-learn pandas

In [4]:

import torch
import numpy as np
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import gc

# 1. ЗАГРУЗКА ДАННЫХ
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/LLMTrace_4class_20k.csv')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:

# Преобразование в бинарную метку
df['binary_label'] = df['label'].apply(lambda x: 0 if x in [0,3] else 1)
df['is_clean'] = df['label'].isin([0,1])

# Удаляем выбросы
df['word_count'] = df['text'].apply(lambda x: len(x.split()))
def remove_outliers(df, genre_col='style', upper_quantile=0.95):
    cleaned = []
    for genre in df[genre_col].unique():
        human_subset = df[(df[genre_col]==genre) & (df['binary_label']==0)]
        if len(human_subset) > 0:
            thresh = human_subset['word_count'].quantile(upper_quantile)
            human_filtered = human_subset[human_subset['word_count'] <= thresh]
            cleaned.append(human_filtered)
        ai_subset = df[(df[genre_col]==genre) & (df['binary_label']==1)]
        cleaned.append(ai_subset)
    return pd.concat(cleaned, ignore_index=True)

df = remove_outliers(df, upper_quantile=0.95)
df.drop(columns=['word_count'], inplace=True)

df['stratify'] = df['binary_label'].astype(str) + '_' + df['style']
train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, random_state=42, stratify=df['stratify']
)
train_full = df.loc[train_idx].copy()
test_full = df.loc[test_idx].copy()
train_full.drop(columns=['stratify'], inplace=True)
test_full.drop(columns=['stratify'], inplace=True)

print(f"Train size: {len(train_full)}, Test size: {len(test_full)}")
print(f"Жанры: {df['style'].unique().tolist()}")



Train size: 9730, Test size: 2433
Жанры: ['literature', 'publicistic', 'scientific']


In [6]:

# 2. РЕАЛИЗАЦИЯ FAST-DETECTGPT
class FastDetectGPT:
    def __init__(self, model_name="Qwen/Qwen2.5-0.5B", max_length=512, device=None):
        self.device = device if device else ("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Используемое устройство: {self.device}")

        print(f"Загрузка модели {model_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
            device_map="auto" if self.device == "cuda" else None,
            trust_remote_code=True
        ).to(self.device)
        self.model.eval()

        self.max_length = max_length
        self.threshold = 0.5

    def compute_criterion(self, text):
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True,
                                max_length=self.max_length).to(self.device)
        input_ids = inputs.input_ids

        with torch.no_grad():
            outputs = self.model(input_ids, labels=input_ids)
            logits = outputs.logits

        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = input_ids[..., 1:].contiguous()

        log_probs = -torch.nn.functional.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
            reduction='none'
        )

        criterion = -log_probs.mean().item()
        return criterion

    def compute_scores(self, texts, batch_size=4):
        """Вычисляет скоры для списка текстов."""
        scores = []
        for i in tqdm(range(0, len(texts), batch_size), desc="Вычисление критериев"):
            batch_texts = texts[i:i+batch_size]
            batch_scores = []
            for text in batch_texts:
                try:
                    score = self.compute_criterion(text)
                    batch_scores.append(score)
                except Exception as e:
                    print(f"Ошибка: {e}")
                    batch_scores.append(0.0)
            scores.extend(batch_scores)
        return np.array(scores)

    def calibrate_threshold(self, X_text, y_true):
        scores = self.compute_scores(X_text)
        best_thresh = 0.5
        best_acc = 0.0

        for thresh in np.arange(0.0, 1.01, 0.01):
            pred = (scores > thresh).astype(int)
            acc = accuracy_score(y_true, pred)
            if acc > best_acc:
                best_acc = acc
                best_thresh = thresh

        self.threshold = best_thresh
        print(f"Калиброванный порог: {self.threshold:.4f} (Accuracy: {best_acc:.4f})")
        return self.threshold

    def predict(self, X_text):
        scores = self.compute_scores(X_text)
        return (scores > self.threshold).astype(int)

# ==================================================
# 3. ФУНКЦИЯ ОЦЕНКИ
# ==================================================
def evaluate_model(detector, X_train, y_train, X_test, y_test, exp_name, train_genre, test_genre):
    print(f"\n>>> {exp_name}: обучение на {train_genre} -> тест на {test_genre}")
    print(f"Калибровка на {len(X_train)} текстах...")
    detector.calibrate_threshold(X_train, y_train)

    print(f"Тестирование на {len(X_test)} текстах...")
    y_pred = detector.predict(X_test)

    return {
        'accuracy': accuracy_score(y_test, y_pred),
        'f1_AI': f1_score(y_test, y_pred, pos_label=1, zero_division=0),
        'f1_Human': f1_score(y_test, y_pred, pos_label=0, zero_division=0),
        'precision_AI': precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        'recall_AI': recall_score(y_test, y_pred, pos_label=1, zero_division=0),
        'experiment': exp_name,
        'genre_train': train_genre,
        'genre_test': test_genre
    }



In [1]:
# 4. ПРОВЕДЕНИЕ ЭКСПЕРИМЕНТОВ
genres = df['style'].unique().tolist()
genre_column = 'style'
results = []


print("ЭКСПЕРИМЕНТЫ 1")
print("="*80)

train_clean = train_full[train_full['is_clean'] == True]
test_clean = test_full[test_full['is_clean'] == True]

# обучение на жанре -> тест на том же жанре / всех жанрах
for genre in genres:
    train_genre = train_clean[train_clean[genre_column] == genre]
    if len(train_genre) < 10:
        continue

    test_genre = test_clean[test_clean[genre_column] == genre]
    if len(test_genre) > 0:
        detector = FastDetectGPT(model_name="Qwen/Qwen2.5-0.5B")
        metrics = evaluate_model(
            detector,
            train_genre['text'].values, train_genre['binary_label'].values,
            test_genre['text'].values, test_genre['binary_label'].values,
            '1A_clean_same_genre', genre, genre
        )
        results.append(metrics)
        del detector
        gc.collect()
        torch.cuda.empty_cache()

    if len(test_clean) > 0:
        detector = FastDetectGPT(model_name="Qwen/Qwen2.5-0.5B")
        metrics = evaluate_model(
            detector,
            train_genre['text'].values, train_genre['binary_label'].values,
            test_clean['text'].values, test_clean['binary_label'].values,
            '1B_clean_cross_genre', genre, 'all'
        )
        results.append(metrics)
        del detector
        gc.collect()
        torch.cuda.empty_cache()

# обучение на всех чистых -> тест на каждом жанре
if len(train_clean) > 0:
    for genre in genres:
        test_genre = test_clean[test_clean[genre_column] == genre]
        if len(test_genre) > 0:
            detector = FastDetectGPT(model_name="Qwen/Qwen2.5-0.5B")
            metrics = evaluate_model(
                detector,
                train_clean['text'].values, train_clean['binary_label'].values,
                test_genre['text'].values, test_genre['binary_label'].values,
                '1C_clean_all_train', 'all', genre
            )
            results.append(metrics)
            del detector
            gc.collect()
            torch.cuda.empty_cache()

print("ЭКСПЕРИМЕНТ 2")
print("="*80)

train_all = train_full
test_all = test_full

# обучение на жанре -> тест на том же жанре / всех жанрах
for genre in genres:
    train_genre = train_all[train_all[genre_column] == genre]
    if len(train_genre) < 10:
        continue

    test_genre = test_all[test_all[genre_column] == genre]
    if len(test_genre) > 0:
        detector = FastDetectGPT(model_name="Qwen/Qwen2.5-0.5B")
        metrics = evaluate_model(
            detector,
            train_genre['text'].values, train_genre['binary_label'].values,
            test_genre['text'].values, test_genre['binary_label'].values,
            '2A_all_same_genre', genre, genre
        )
        results.append(metrics)
        del detector
        gc.collect()
        torch.cuda.empty_cache()

    if len(test_all) > 0:
        detector = FastDetectGPT(model_name="Qwen/Qwen2.5-0.5B")
        metrics = evaluate_model(
            detector,
            train_genre['text'].values, train_genre['binary_label'].values,
            test_all['text'].values, test_all['binary_label'].values,
            '2B_all_cross_genre', genre, 'all'
        )
        results.append(metrics)
        del detector
        gc.collect()
        torch.cuda.empty_cache()

# обучение на всех данных -> тест на каждом жанре
for genre in genres:
    test_genre = test_all[test_all[genre_column] == genre]
    if len(test_genre) > 0:
        detector = FastDetectGPT(model_name="Qwen/Qwen2.5-0.5B")
        metrics = evaluate_model(
            detector,
            train_all['text'].values, train_all['binary_label'].values,
            test_genre['text'].values, test_genre['binary_label'].values,
            '2C_all_train', 'all', genre
        )
        results.append(metrics)
        del detector
        gc.collect()
        torch.cuda.empty_cache()

print("ЭКСПЕРИМЕНТЫ 3")
print("="*80)

test_unclean = test_full[test_full['is_clean'] == False]

if len(test_unclean) > 0:
    # обучение на чистом жанре -> тест на нечистых того же жанра / всех нечистых
    for genre in genres:
        train_genre = train_clean[train_clean[genre_column] == genre]
        if len(train_genre) < 10:
            continue

        test_unclean_genre = test_unclean[test_unclean[genre_column] == genre]
        if len(test_unclean_genre) > 0:
            detector = FastDetectGPT(model_name="Qwen/Qwen2.5-0.5B")
            metrics = evaluate_model(
                detector,
                train_genre['text'].values, train_genre['binary_label'].values,
                test_unclean_genre['text'].values, test_unclean_genre['binary_label'].values,
                '3A_clean_same_genre_test_unclean', genre, f'unclean_{genre}'
            )
            results.append(metrics)
            del detector
            gc.collect()
            torch.cuda.empty_cache()

        if len(test_unclean) > 0:
            detector = FastDetectGPT(model_name="Qwen/Qwen2.5-0.5B")
            metrics = evaluate_model(
                detector,
                train_genre['text'].values, train_genre['binary_label'].values,
                test_unclean['text'].values, test_unclean['binary_label'].values,
                '3B_clean_same_genre_test_all_unclean', genre, 'all_unclean'
            )
            results.append(metrics)
            del detector
            gc.collect()
            torch.cuda.empty_cache()

    # обучение на всех чистых -> тест на нечистых по жанрам
    if len(train_clean) > 0:
        for genre in genres:
            test_unclean_genre = test_unclean[test_unclean[genre_column] == genre]
            if len(test_unclean_genre) > 0:
                detector = FastDetectGPT(model_name="Qwen/Qwen2.5-0.5B")
                metrics = evaluate_model(
                    detector,
                    train_clean['text'].values, train_clean['binary_label'].values,
                    test_unclean_genre['text'].values, test_unclean_genre['binary_label'].values,
                    '3C_clean_all_train_test_unclean_per_genre', 'all_clean', f'unclean_{genre}'
                )
                results.append(metrics)
                del detector
                gc.collect()
                torch.cuda.empty_cache()

# 5. ВЫВОД РЕЗУЛЬТАТОВ
# ==================================================
df_results = pd.DataFrame(results)
df_results['method'] = 'FastDetectGPT'

print("ИТОГОВЫЕ РЕЗУЛЬТАТЫ Fast-DetectGPT")
print("="*80)
print(df_results.to_string(index=False))



NameError: name 'df' is not defined